# 1. Setup Lingkungan & Ekstraksi Gambar Sintetis
Jalankan sel ini untuk mengekstrak gambar dari `.pkl` StyleGAN2.

In [ ]:
!pip install click requests tqdm pyspng ninja imageio-ffmpeg==0.4.3
!git clone https://github.com/NVlabs/stylegan2-ada-pytorch.git

import os
import sys
sys.path.insert(0, '/kaggle/working/stylegan2-ada-pytorch')

import pickle
import torch
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

# ---- AUTO-SEARCH PKL TERBAIK ----
print("Mencari file PKL terbaik (000700.pkl)...")
NETWORK_PKL = None
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if '000700.pkl' in f:
            NETWORK_PKL = os.path.join(root, f)
            break
    if NETWORK_PKL:
        break

if not NETWORK_PKL:
    raise FileNotFoundError("ERROR: File snapshot yang mengandung '000700.pkl' tidak ditemukan di Kaggle Input! Pastikan Anda sudah mengunggah dataset PKL.")

print(f"File PKL ditemukan di: {NETWORK_PKL}")

TARGET_SINTETIS = 666
OUT_DIR_SINTETIS = '/kaggle/working/dataset/sintetis_positif'
os.makedirs(OUT_DIR_SINTETIS, exist_ok=True)

print(f"Loading network...")
device = torch.device('cuda')
with open(NETWORK_PKL, 'rb') as f:
    G = pickle.load(f)['G_ema'].to(device)

print(f"Generating {TARGET_SINTETIS} synthetic images...")
for i in tqdm(range(TARGET_SINTETIS)):
    z = torch.randn([1, G.z_dim]).to(device)
    img = G(z, None)
    img = (img.permute(0, 2, 3, 1) * 127.5 + 128).clamp(0, 255).to(torch.uint8)
    img_np = img[0].cpu().numpy()
    Image.fromarray(img_np, 'RGB').save(os.path.join(OUT_DIR_SINTETIS, f"syn_pos_{i:04d}.png"))

print("Ekstraksi selesai!")

# 2. Pembagian Dataset (Zero Data-Leakage)
Membagi dataset menjadi Train, Val, Test untuk Skenario A, dan menyuntikkan Sintetis ke Skenario B.

In [ ]:
import os
import shutil
import random

# ---- AUTO-SEARCH DATASET ASLI ----
print("Mencari folder 'positif' dan 'negatif'...")
DIR_POS_ASLI = None
DIR_NEG_ASLI = None

for root, dirs, files in os.walk('/kaggle/input'):
    folder_name = os.path.basename(root).lower()
    if folder_name == 'positif':
        DIR_POS_ASLI = root
    elif folder_name == 'negatif':
        DIR_NEG_ASLI = root

if not DIR_POS_ASLI or not DIR_NEG_ASLI:
    raise FileNotFoundError("ERROR: Folder bernama 'positif' atau 'negatif' tidak ditemukan di /kaggle/input! Pastikan struktur zip Anda benar.")

print(f"Folder Positif Asli ditemukan di: {DIR_POS_ASLI}")
print(f"Folder Negatif Asli ditemukan di: {DIR_NEG_ASLI}")

TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.8, 0.1, 0.1

def split_and_copy(files, class_name, scenario_dir):
    n = len(files)
    n_train = int(n * TRAIN_RATIO)
    n_val = int(n * VAL_RATIO)
    
    train_files = files[:n_train]
    val_files = files[n_train:n_train+n_val]
    test_files = files[n_train+n_val:]
    
    def copy_files(file_list, split_name):
        dest_dir = os.path.join(scenario_dir, split_name, class_name)
        os.makedirs(dest_dir, exist_ok=True)
        for f in file_list:
            shutil.copy(f, dest_dir)
            
    copy_files(train_files, 'train')
    copy_files(val_files, 'val')
    copy_files(test_files, 'test')
    return train_files

pos_files = [os.path.join(DIR_POS_ASLI, f) for f in os.listdir(DIR_POS_ASLI) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
neg_files = [os.path.join(DIR_NEG_ASLI, f) for f in os.listdir(DIR_NEG_ASLI) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

random.seed(42)
random.shuffle(pos_files)
random.shuffle(neg_files)

print("\nMempersiapkan Skenario A (Baseline)...")
DIR_SKENARIO_A = '/kaggle/working/Skenario_A'
train_pos_a = split_and_copy(pos_files, 'positif', DIR_SKENARIO_A)
split_and_copy(neg_files, 'negatif', DIR_SKENARIO_A)

print("Mempersiapkan Skenario B (Augmented)...")
DIR_SKENARIO_B = '/kaggle/working/Skenario_B'
shutil.copytree(DIR_SKENARIO_A, DIR_SKENARIO_B, dirs_exist_ok=True)

# Hapus Train Positif Asli di Skenario B, ganti dengan Sintetis
print("Menyuntikkan citra sintetis ke Skenario B (Train saja)...")
dest_train_pos_b = os.path.join(DIR_SKENARIO_B, 'train', 'positif')
shutil.rmtree(dest_train_pos_b)
os.makedirs(dest_train_pos_b, exist_ok=True)

syn_files = [os.path.join(OUT_DIR_SINTETIS, f) for f in os.listdir(OUT_DIR_SINTETIS)]
for f in syn_files:
    shutil.copy(f, dest_train_pos_b)

print("Distribusi Data Selesai!")

# 3. Pelatihan Model DenseNet121
Melatih dan mengevaluasi langsung Skenario A kemudian dilanjutkan dengan Skenario B.

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

IMG_SIZE = (128, 128)
BATCH_SIZE = 32
EPOCHS = 20

def build_model():
    base_model = DenseNet121(weights='imagenet', include_top=False, input_shape=(128, 128, 3))
    for layer in base_model.layers:
        layer.trainable = False
        
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.5)(x)
    x = Dense(128, activation='relu')(x)
    predictions = Dense(1, activation='sigmoid')(x) 
    
    model = Model(inputs=base_model.input, outputs=predictions)
    model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
    return model

def train_and_evaluate(scenario_dir, scenario_name):
    print(f"\n{'='*50}\nMEMULAI PELATIHAN: {scenario_name}\n{'='*50}")
    
    train_dir = os.path.join(scenario_dir, 'train')
    val_dir = os.path.join(scenario_dir, 'val')
    test_dir = os.path.join(scenario_dir, 'test')
    
    datagen = ImageDataGenerator(rescale=1./255)
    
    train_generator = datagen.flow_from_directory(
        train_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary')
    
    val_generator = datagen.flow_from_directory(
        val_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary', shuffle=False)
        
    test_generator = datagen.flow_from_directory(
        test_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary', shuffle=False)
        
    model = build_model()
    
    history = model.fit(
        train_generator,
        epochs=EPOCHS,
        validation_data=val_generator,
        verbose=1
    )
    
    print(f"\n--- Evaluasi Test Set ({scenario_name}) ---")
    predictions = model.predict(test_generator)
    y_pred = (predictions > 0.5).astype(int).reshape(-1)
    y_true = test_generator.classes
    
    print(classification_report(y_true, y_pred, target_names=list(test_generator.class_indices.keys())))
    
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=test_generator.class_indices.keys(),
                yticklabels=test_generator.class_indices.keys())
    plt.title(f'Confusion Matrix - {scenario_name}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()

train_and_evaluate(DIR_SKENARIO_A, 'Skenario A (Baseline)')
train_and_evaluate(DIR_SKENARIO_B, 'Skenario B (Augmented)')